# TFG V7: Quantum Walk with Defect for Contiguous Free-Region Search

This notebook implements a **continuous-time quantum walk with local defects** for the contiguous free-region search problem on multidimensional discrete grids.

The goal is to find all window-start positions whose associated window is entirely free. The quantum system lives only on the candidate window-start graph, so the quantum circuit uses a single index register and compiles the classically known valid windows as local energy defects.


## Problem Definition

Given a $D$-dimensional binary grid of sizes

$$N=(N_1,\ldots,N_D),$$

where occupied cells are marked by 1 and free cells by 0, we search for start positions $i$ such that a contiguous window

$$M=(M_1,\ldots,M_D)$$

is entirely free. The set of candidate starts is

$$\prod_{d=1}^D \{0,\ldots,N_d-M_d\},$$

with total size

$$W=\prod_d (N_d-M_d+1).$$

These starts form a graph: two starts are adjacent if their coordinates differ by exactly 1 in exactly one dimension. The graph is a line in 1D, a rectangular lattice in 2D, and a Cartesian grid in higher dimensions.


## Theoretical Background

### Discrete-time vs continuous-time quantum walks

A **discrete-time quantum walk** usually alternates two operations: a coin operation and a shift operation. The coin decides which direction the walker should move, and the shift moves the walker through the graph. This typically requires an explicit coin register.

A **continuous-time quantum walk** has no coin register. The walker evolves directly under a Hamiltonian,

$$|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle.$$

In this notebook, the base Hamiltonian is built from the adjacency matrix of the window-start graph. The off-diagonal entries generate hopping between neighbouring candidate windows.

### Defect mechanism

The valid all-zero windows are marked by adding a local energy perturbation:

$$H_{\mathrm{defect}} = -\gamma A - \lambda \sum_{k\in V} |k\rangle\langle k|,$$

where $A$ is the adjacency matrix, $V$ is the set of valid window indices, $\gamma$ is the hopping rate, and $\lambda>0$ is the defect strength. The minus sign in front of $A$ is the standard spatial-search convention: the uniform-like low-energy state of the graph can then couple resonantly to the lowered marked states.

The negative diagonal term lowers the energy of valid nodes. Spectrally, this can pull bound states out of the bulk band and localise amplitude near the marked nodes. Starting from a uniform superposition, the walk can transfer probability into the valid subspace at special times.

### Connection to V4

V4 can be interpreted as a first-order Trotterisation of this continuous-time quantum walk. The cost-phase oracle approximates

$$e^{i\lambda P_V\Delta t},$$

while the local mixer approximates

$$e^{+i\gamma A\Delta t}.$$

The V4 layer is therefore a digital approximation to the continuous-time defect walk. The quantum-walk framework gives principled values for the phase and mixer angles through $\lambda$, $\gamma$, and $t$, instead of requiring a variational search over $\theta$ and $\beta$.


## Requirements and Imports

This cell imports the numerical, plotting, and Qiskit objects used throughout the notebook. The plotting backend is configured for file generation because all figures are saved to disk.


In [20]:

# pip install qiskit qiskit-aer qiskit-ibm-runtime numpy matplotlib

import os
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

from math import prod
from pathlib import Path
import csv
import re
import time

import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, PhaseGate, UnitaryGate

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

VALID_COLOR = "#2ecc71"
INVALID_COLOR = "#e74c3c"
BASELINE_COLOR = "0.45"

V7_OUTPUT_DIR = Path("TFG_V7_Analysis")
V7_OUTPUT_DIR.mkdir(exist_ok=True)

# V7 uses the earliest time whose success probability is within this
# absolute tolerance of the best scanned peak. This keeps the walk fast
# while preserving a strong success-probability advantage.
V7_PEAK_TOLERANCE = 0.05

# If True, each case scans only times t <= r_Grover = (pi/4)*sqrt(W/K).
# This forces the quantum-walk search to use no more time/iteration scale
# than the standard Grover estimate for the same valid fraction.
V7_ENFORCE_GROVER_TIME_CAP = True
V7_DEFAULT_T_MAX = 200.0
V7_T_STEPS = 4000

print("qiskit version:", qiskit.__version__)


qiskit version: 2.3.1


## Utility Functions

The following utility functions are copied directly from `TFG_V4.ipynb`. They define the grid geometry, the classical cost function, the valid-window set, Gray ordering, and the initial uniform superposition over valid index states.


In [21]:

# =========================================================
# ND geometry and classical utilities
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


In [22]:

# =========================================================
# Circuit construction utility copied from V4
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


## Case Studies

The ten case studies below are exactly the same instances used in `TFG_V4.ipynb`: five 1D cases, four 2D cases, and one 3D case. This makes the V7 quantum-walk results directly comparable with the V4 cost-phase plus mixer experiments.


In [23]:

V4_CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in the manual main experiment.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3, "beta": 0.60, "mixer_method": "local_geometric",
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2, "beta": 0.28, "mixer_method": "local_geometric",
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3, "beta": 0.22, "mixer_method": "local_geometric",
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },
]


## Context Builders and Plot Helpers

The context builder converts each V4 case into the classical data needed by the quantum walk: starts, costs, valid indices, $W$, and the uniform success baseline $K/W$. The plotting helper saves each figure as both PDF and PNG.


In [24]:

def qw_slug(text):
    """Return a filesystem-safe lowercase slug for figure filenames."""
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(text)).strip("_").lower()


def save_qw_figure(fig, stem):
    """Save a Matplotlib figure as PDF and PNG in the V7 analysis folder."""
    # Guardamos cada figura solo en la carpeta V7 del experimento.
    V7_OUTPUT_DIR.mkdir(exist_ok=True)
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def qw_case_context(case):
    """Build the classical search-space context for one V4 case study."""
    # Validamos dimensiones y precalculamos costes clasicos por ventana.
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case = [
        compute_window_cost_classical(grid_bits_case, s, N_case, M_case)
        for s in starts_case
    ]
    valid_indices_case = [i for i, c in enumerate(costs_case) if c == 0]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows; change occupied_coords.")
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    if W_case > 2**IDX_case:
        raise ValueError(f"W={W_case} does not fit in IDX={IDX_case} qubits.")
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "valid_indices": valid_indices_case,
        "W": W_case,
        "K": len(valid_indices_case),
        "IDX": IDX_case,
        "P_uniform": len(valid_indices_case) / W_case,
    }


CASE_CONTEXTS = [qw_case_context(case) for case in V4_CASE_STUDIES]
print(f"Loaded {len(CASE_CONTEXTS)} case contexts.")
for ctx in CASE_CONTEXTS:
    print(f"{ctx['name']} | W={ctx['W']} | K={ctx['K']} | P_uniform={ctx['P_uniform']:.4f}")


Loaded 10 case contexts.
01_1d_tiny_single_gap | W=5 | K=1 | P_uniform=0.2000
02_1d_main_reference | W=7 | K=2 | P_uniform=0.2857
03_1d_two_free_regions | W=8 | K=3 | P_uniform=0.3750
04_1d_clustered_medium | W=14 | K=4 | P_uniform=0.2857
05_1d_long_clustered_blocks | W=29 | K=11 | P_uniform=0.3793
06_2d_tiny_corner_block | W=4 | K=1 | P_uniform=0.2500
07_2d_small_diagonal_block | W=9 | K=1 | P_uniform=0.1111
08_2d_medium_clustered_obstacles | W=16 | K=9 | P_uniform=0.5625
09_2d_rectangular_window | W=20 | K=8 | P_uniform=0.4000
10_3d_small_clustered_obstacles | W=18 | K=12 | P_uniform=0.6667


## Quantum-Walk Core Functions

These functions build the window-start graph, its Laplacian, the defect Hamiltonian, and the exact continuous-time evolution. The time search is classical and uses one Hermitian diagonalisation followed by a fully vectorised evaluation of $P_{valid}(t)$.


In [25]:
def build_adjacency_matrix(starts, N):
    """Build the symmetric adjacency matrix of the window-start graph.

    Two window starts are adjacent when they differ by exactly one lattice
    step in one coordinate and are equal in all other coordinates. The
    dictionary lookup makes neighbour discovery O(1) per candidate neighbour.
    """
    # Mapeamos coordenadas a indices para buscar vecinos en tiempo constante.
    starts = [tuple(start) for start in starts]
    W = len(starts)
    D = len(N)
    start_to_index = {start: i for i, start in enumerate(starts)}
    A = np.zeros((W, W), dtype=np.float64)

    for i, start in enumerate(starts):
        for d in range(D):
            for step in (-1, 1):
                neighbour = list(start)
                neighbour[d] += step
                neighbour = tuple(neighbour)
                j = start_to_index.get(neighbour)
                if j is not None:
                    A[i, j] = 1.0
                    A[j, i] = 1.0

    return A


def build_graph_laplacian(A):
    """Return the unnormalised and normalised graph Laplacians.

    The unnormalised Laplacian is L = D - A. The normalised Laplacian is
    D^{-1/2} L D^{-1/2}, with zero inverse degree assigned to isolated nodes.
    """
    # Calculamos grados y evitamos divisiones por cero en nodos aislados.
    A = np.asarray(A, dtype=np.float64)
    degrees = A @ np.ones(A.shape[0], dtype=np.float64)
    L = np.diag(degrees) - A
    inv_sqrt_degrees = np.zeros_like(degrees)
    nonzero = degrees > 0
    inv_sqrt_degrees[nonzero] = 1.0 / np.sqrt(degrees[nonzero])
    D_inv_sqrt = np.diag(inv_sqrt_degrees)
    L_norm = D_inv_sqrt @ L @ D_inv_sqrt
    return L, L_norm


def build_defect_hamiltonian(A, valid_indices, lam, gamma=1.0, hopping_sign=-1.0):
    """Build the CTQW defect Hamiltonian.

    The default is the spatial-search sign convention
    H = -gamma*A - lam*sum_k |k><k|.  The negative hopping sign makes the
    uniform graph mode a low-energy reference state, so the negative defect can
    create an avoided crossing with marked-node states.  Set hopping_sign=+1
    only to reproduce the earlier diagnostic convention H = +gamma*A - lam*P.
    """
    # Copiamos A para no modificar la matriz de adyacencia original.
    H = float(hopping_sign) * float(gamma) * np.asarray(A, dtype=np.float64).copy()
    for k in valid_indices:
        H[int(k), int(k)] -= float(lam)
    if not np.allclose(H, H.conj().T, atol=1e-12):
        raise ValueError("H_defect is not Hermitian.")
    return H


def find_optimal_time(H, valid_indices, W, t_max=200, t_steps=4000, peak_tolerance=0.0):
    """Find a high-quality time for the valid-subspace probability.

    The Hamiltonian is diagonalised once with np.linalg.eigh. The full time
    curve is evaluated by broadcasting exp(-i E_k t). If peak_tolerance > 0,
    the returned t_star is the earliest sampled time whose probability is
    within peak_tolerance of the maximum. This avoids selecting very late
    recurrences when an almost equally good first peak exists.
    """
    # Caso trivial: un unico nodo. Si es valido, la probabilidad es constante.
    if W == 1:
        t_values = np.linspace(0.0, float(t_max), int(t_steps))
        p_value = 1.0 if 0 in set(valid_indices) else 0.0
        return 0.0, p_value, t_values, np.full_like(t_values, p_value, dtype=float)

    valid_indices = list(map(int, valid_indices))
    if not valid_indices:
        raise ValueError("valid_indices must not be empty.")

    eigenvalues, eigenvectors = np.linalg.eigh(np.asarray(H, dtype=np.complex128))
    psi0 = np.ones(int(W), dtype=complex) / np.sqrt(W)
    coeffs = eigenvectors.conj().T @ psi0
    t_values = np.linspace(0.0, float(t_max), int(t_steps))

    phases = np.exp(-1j * eigenvalues[:, None] * t_values[None, :])
    psi_t = eigenvectors @ (phases * coeffs[:, None])
    p_valid_curve = np.sum(np.abs(psi_t[valid_indices, :])**2, axis=0)

    max_idx = int(np.argmax(p_valid_curve))
    max_p = float(p_valid_curve[max_idx])
    if peak_tolerance and peak_tolerance > 0:
        threshold = max(0.0, max_p - float(peak_tolerance))
        candidate_indices = np.flatnonzero(p_valid_curve >= threshold)
        best_idx = int(candidate_indices[0]) if len(candidate_indices) else max_idx
    else:
        best_idx = max_idx

    t_star = float(t_values[best_idx])
    p_star = float(p_valid_curve[best_idx])
    return t_star, p_star, t_values, p_valid_curve


def scan_lambda(H_adj, valid_indices, W, lambda_values=None, t_max=200, t_steps=4000,
                gamma=1.0, hopping_sign=-1.0, peak_tolerance=V7_PEAK_TOLERANCE):
    """Scan defect strengths and return (lambda, t_star, p_star) rows."""
    # La rejilla V7 combina valores lineales y geometricos para no perder resonancias estrechas.
    if lambda_values is None:
        lambda_values = v7_lambda_grid(W) if "v7_lambda_grid" in globals() else np.linspace(0.1, 5.0 * np.sqrt(W), 30)
    rows = []
    for lam in lambda_values:
        H = build_defect_hamiltonian(
            H_adj, valid_indices, lam=float(lam), gamma=gamma, hopping_sign=hopping_sign
        )
        t_star, p_star, _t_values, _p_curve = find_optimal_time(
            H, valid_indices, W, t_max=t_max, t_steps=t_steps, peak_tolerance=peak_tolerance
        )
        rows.append((float(lam), float(t_star), float(p_star)))
    return rows


## Unit Tests for the Graph Construction

Before using the graph as a Hamiltonian, we check the two basic structural properties required of an undirected simple graph: symmetry and zero diagonal.


In [26]:

for ctx in CASE_CONTEXTS:
    A_test = build_adjacency_matrix(ctx["starts"], ctx["N"])
    assert A_test.shape == (ctx["W"], ctx["W"])
    assert np.allclose(A_test, A_test.T), f"Adjacency is not symmetric for {ctx['name']}"
    assert np.allclose(np.diag(A_test), 0.0), f"Adjacency diagonal is not zero for {ctx['name']}"
print("Adjacency unit tests passed for all V4 case studies.")


Adjacency unit tests passed for all V4 case studies.


## Exact Quantum Circuit Construction

The circuit uses only one quantum register, `i`, holding the window-start index. The continuous-time evolution is compiled exactly as a dense `UnitaryGate` after diagonalising the defect Hamiltonian. States outside the valid index range $0,\ldots,W-1$ remain untouched by an identity block.


In [27]:
def build_quantum_walk_circuit(N, M, occupied_coords, lam, t_star, gamma=1.0, hopping_sign=-1.0):
    """Build the exact continuous-time defect quantum-walk circuit.

    The circuit prepares the uniform superposition over W candidate starts,
    embeds exp(-i H_defect t_star) into the 2**IDX-dimensional index register,
    and returns both the circuit and metadata needed for analysis.
    """
    # Construimos todo el problema clasico para compilar el Hamiltoniano exacto.
    validate_problem(N, M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    if W > 2**IDX:
        raise ValueError(f"W={W} does not fit in IDX={IDX} qubits.")

    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valid_indices = [i for i, cost in enumerate(costs) if cost == 0]
    if not valid_indices:
        raise ValueError("The instance has no valid all-zero windows.")

    A = build_adjacency_matrix(starts, N)
    H_defect = build_defect_hamiltonian(A, valid_indices, lam=lam, gamma=gamma, hopping_sign=hopping_sign)

    eigenvalues, eigenvectors = np.linalg.eigh(H_defect)
    phases = np.exp(-1j * eigenvalues * float(t_star))
    U_walk_W = eigenvectors @ np.diag(phases) @ eigenvectors.conj().T

    dim_full = 2**IDX
    U_full = np.eye(dim_full, dtype=complex)
    U_full[:W, :W] = U_walk_W
    assert np.allclose(U_full @ U_full.conj().T, np.eye(dim_full), atol=1e-8)

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    if W > 1:
        sign_label = "-A" if hopping_sign < 0 else "+A"
        label = f"QW({sign_label}, t={float(t_star):.2f}, lam={float(lam):.3f})"
        qc.append(UnitaryGate(U_full, label=label), list(idx))

    metadata = {
        "N": list(N),
        "M": list(M),
        "occupied_coords": [normalize_coord(c, len(N)) for c in occupied_coords],
        "W": W,
        "K": len(valid_indices),
        "IDX": IDX,
        "valid_indices": valid_indices,
        "starts": starts,
        "grid_bits": grid_bits,
        "costs": costs,
        "lam": float(lam),
        "gamma": float(gamma),
        "hopping_sign": float(hopping_sign),
        "t_star": float(t_star),
        "P_uniform": len(valid_indices) / W,
        "H_defect": H_defect,
        "A": A,
    }
    return qc, metadata


def index_probs_from_statevector(sv, W, IDX):
    """Extract probabilities over the first W index states from a statevector."""
    # El registro cuantico solo contiene idx; los primeros W estados son los candidatos fisicos.
    data = np.asarray(sv.data, dtype=complex)
    outside_weight = float(np.sum(np.abs(data[W:])**2)) if W < len(data) else 0.0
    if outside_weight >= 1e-10:
        raise ValueError(f"Unexpected probability {outside_weight:.3e} in padded index states.")
    return np.abs(data[:W])**2


def run_qw_case(ctx, lam=None, gamma=1.0, hopping_sign=-1.0, t_max=200, t_steps=4000,
                peak_tolerance=V7_PEAK_TOLERANCE):
    """Run the full exact-defect quantum-walk pipeline for one case context."""
    # Si no se proporciona lambda, usamos la heuristica inicial sqrt(W)*gamma.
    W = int(ctx["W"])
    valid_indices = list(ctx["valid_indices"])
    A = build_adjacency_matrix(ctx["starts"], ctx["N"])
    if lam is None:
        lam = np.sqrt(W) * float(gamma)
    H_defect = build_defect_hamiltonian(
        A, valid_indices, lam=lam, gamma=gamma, hopping_sign=hopping_sign
    )
    t_star, p_star, t_values, p_valid_curve = find_optimal_time(
        H_defect, valid_indices, W, t_max=t_max, t_steps=t_steps, peak_tolerance=peak_tolerance
    )

    qc, metadata = build_quantum_walk_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"], lam=lam, t_star=t_star,
        gamma=gamma, hopping_sign=hopping_sign
    )
    sv = Statevector.from_instruction(qc)
    probs = index_probs_from_statevector(sv, metadata["W"], metadata["IDX"])
    p_valid_circuit = float(np.sum(probs[valid_indices]))
    assert abs(p_valid_circuit - p_star) <= 1e-4, (
        f"Circuit/model mismatch: circuit={p_valid_circuit}, model={p_star}"
    )

    r_grover = np.pi / 4.0 * np.sqrt(W / max(1, len(valid_indices)))
    result = {
        "case": ctx["name"],
        "ctx": ctx,
        "qc": qc,
        "metadata": metadata,
        "A": A,
        "H_defect": H_defect,
        "lam": float(lam),
        "gamma": float(gamma),
        "hopping_sign": float(hopping_sign),
        "t_star": float(t_star),
        "P_valid": p_valid_circuit,
        "P_valid_model": float(p_star),
        "P_uniform": float(ctx["P_uniform"]),
        "probs": probs,
        "t_values": t_values,
        "p_valid_curve": p_valid_curve,
        "r_Grover": float(r_grover),
    }
    sign_label = "-A" if hopping_sign < 0 else "+A"
    print(
        f"{ctx['name']} | H={sign_label}-lambdaP | W={W} | K={len(valid_indices)} | "
        f"lam*={float(lam):.4f} | t*={t_star:.4f} | "
        f"P_valid={p_valid_circuit:.4f} | vs Grover r={r_grover:.4f}"
    )
    return result


## Plotting Utilities for the Analysis Cells

These helper functions keep the analysis cells short while ensuring consistent colours, labels, figure sizes, and saved outputs.


In [ ]:
def draw_time_evolution(ax, result, free_curve=None):
    """Draw P_valid(t) for the defect walk and optionally the free walk."""
    sign_label = "-A-lambdaP" if result.get("hopping_sign", -1.0) < 0 else "+A-lambdaP"
    ax.plot(result["t_values"], result["p_valid_curve"], color=VALID_COLOR, linewidth=2.0, label=f"defect walk ({sign_label})")
    if free_curve is not None:
        t_free, p_free = free_curve
        ax.plot(t_free, p_free, color="black", linestyle="-.", linewidth=1.5, alpha=0.8, label="free walk")
    ax.axvline(result["t_star"], color="black", linestyle=":", linewidth=1.5, label=f"t*={result['t_star']:.2f}")
    ax.axhline(result["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.set_xlabel("time t")
    ax.set_ylabel("P_valid(t)")
    ax.legend(fontsize=10)


def spectrum_bound_state_data(result):
    """Compute defect eigenvalues and identify eigenvalues below the matching free-walk band."""
    # La banda de referencia usa el mismo signo de hopping que el Hamiltoniano principal.
    H_defect = result["H_defect"]
    A = result["A"]
    gamma = result["gamma"]
    hopping_sign = result.get("hopping_sign", -1.0)
    defect_eigs = np.linalg.eigvalsh(H_defect)
    free_eigs = np.linalg.eigvalsh(hopping_sign * gamma * A)
    bulk_edge = float(np.min(free_eigs))
    bound_mask = defect_eigs < bulk_edge - 1e-8
    return defect_eigs, bulk_edge, bound_mask


def draw_spectrum(ax, result):
    """Draw the real-line eigenvalue spectrum of H_defect."""
    eigs, bulk_edge, bound_mask = spectrum_bound_state_data(result)
    y = np.zeros_like(eigs)
    ax.scatter(eigs[~bound_mask], y[~bound_mask], color=BASELINE_COLOR, s=38, label="bulk")
    if np.any(bound_mask):
        ax.scatter(eigs[bound_mask], y[bound_mask], color=VALID_COLOR, marker="*", s=160, label="bound state")
        nearest_bound = float(np.max(eigs[bound_mask]))
        gap = bulk_edge - nearest_bound
        ax.plot([nearest_bound, bulk_edge], [0.08, 0.08], color="black", linewidth=1.4)
        ax.text((nearest_bound + bulk_edge) / 2, 0.11, f"gap={gap:.3f}", ha="center", fontsize=10)
        ax.annotate("bound state", xy=(nearest_bound, 0.0), xytext=(nearest_bound, 0.25),
                    arrowprops={"arrowstyle": "->", "linewidth": 1.0}, fontsize=10)
    else:
        ax.text(0.02, 0.85, "no isolated bound state", transform=ax.transAxes, fontsize=10)
    ax.axvline(bulk_edge, color="black", linestyle="--", linewidth=1.0, label="free band edge")
    ax.set_yticks([])
    ax.set_xlabel("eigenvalue")
    ax.set_ylim(-0.2, 0.45)
    ax.legend(fontsize=9)


def draw_distribution(ax, result):
    """Draw the optimal index probability distribution with valid/invalid colours."""
    W = result["ctx"]["W"]
    valid = set(result["ctx"]["valid_indices"])
    colors = [VALID_COLOR if i in valid else INVALID_COLOR for i in range(W)]
    x = np.arange(W)
    ax.bar(x, result["probs"], color=colors, alpha=0.9)
    ax.axhline(1.0 / W, color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="1/W")
    ax.set_xlabel("window index i")
    ax.set_ylabel("probability")
    ax.legend(fontsize=10)


def best_lambda_from_scan(lambda_scan_rows, p_tolerance=V7_PEAK_TOLERANCE):
    """Choose the earliest near-optimal lambda-scan row.

    We first find the maximum scanned probability, then accept all lambdas
    within p_tolerance of it and choose the one with the smallest t_star.
    V7 uses this as a probability-time tradeoff, so we do not pay for late
    recurrence peaks that only improve P_valid slightly.
    """
    max_p = max(row[2] for row in lambda_scan_rows)
    candidates = [row for row in lambda_scan_rows if row[2] >= max_p - float(p_tolerance)]
    return min(candidates, key=lambda row: (row[1], -row[2], row[0]))


def v7_lambda_grid(W):
    """Return the V7 lambda scan grid used for the main defect runs."""
    # Mezclamos puntos geometricos y lineales para capturar resonancias sin hacer la celda lenta.
    upper = 5.0 * np.sqrt(W)
    weak_to_mid = np.geomspace(0.05, upper, 32)
    linear = np.linspace(0.1, upper, 36)
    heuristic_band = np.sqrt(W) * np.linspace(0.25, 1.75, 20)
    return np.unique(np.r_[weak_to_mid, linear, heuristic_band])


def v7_diagnostic_lambda_grid(W):
    """Return a wider lambda grid including zero and weak-defect values."""
    # Esta rejilla diagnostica comprueba si el optimo esta pegado a lambda=0.
    weak = np.geomspace(1e-4, 0.2, 24)
    regular = np.linspace(0.25, 5.0 * np.sqrt(W), 30)
    return np.unique(np.r_[0.0, weak, regular])


def build_old_sign_hamiltonian(A, valid_indices, lam, gamma=1.0):
    """Build the previous diagnostic convention H = +gamma*A - lam*P_V."""
    # Se conserva solo para comparar contra la version corregida.
    return build_defect_hamiltonian(A, valid_indices, lam=lam, gamma=gamma, hopping_sign=+1.0)


def scan_lambda_with_builder(A, valid_indices, W, lambda_values, builder, t_max=200, t_steps=4000,
                             peak_tolerance=V7_PEAK_TOLERANCE):
    """Scan lambda values using a supplied Hamiltonian builder."""
    rows = []
    for lam in lambda_values:
        H = builder(A, valid_indices, float(lam))
        t_star, p_star, _t_values, _p_curve = find_optimal_time(
            H, valid_indices, W, t_max=t_max, t_steps=t_steps, peak_tolerance=peak_tolerance
        )
        rows.append((float(lam), float(t_star), float(p_star)))
    return rows


def free_walk_curve(ctx, A, t_max=200, t_steps=4000, hopping_sign=-1.0):
    """Return the lambda=0 time curve for the selected hopping-sign convention."""
    H_free = build_defect_hamiltonian(
        A, ctx["valid_indices"], lam=0.0, gamma=1.0, hopping_sign=hopping_sign
    )
    t_star, p_star, t_values, p_curve = find_optimal_time(
        H_free, ctx["valid_indices"], ctx["W"], t_max=t_max, t_steps=t_steps, peak_tolerance=V7_PEAK_TOLERANCE
    )
    return {
        "t_star": float(t_star),
        "P_free": float(p_star),
        "t_values": t_values,
        "p_curve": p_curve,
    }



def grover_iteration_scale(W, K):
    """Return the usual Grover time/iteration scale (pi/4)*sqrt(W/K)."""
    # Esta escala se usa como cota opcional para comparar complejidad temporal.
    if K <= 0:
        raise ValueError("K must be positive for Grover scaling.")
    return float(np.pi / 4.0 * np.sqrt(float(W) / float(K)))


def grover_success_probability(W, K, r_value=None):
    """Return ideal continuous Grover success available within r_value.

    The textbook expression is sin^2((2r+1)theta), with
    sin(theta)=sqrt(K/W). For a theoretical face-off we allow the ideal
    continuous first peak if it lies within the Grover time scale; otherwise we
    report the endpoint probability. This avoids misleading overshoot artefacts
    for cases with a large valid fraction K/W.
    """
    # Comparamos contra el limite teorico ideal de Grover dentro de su escala temporal.
    if K <= 0:
        raise ValueError("K must be positive for Grover probability.")
    theta_g = np.arcsin(np.sqrt(float(K) / float(W)))
    if r_value is None:
        r_value = grover_iteration_scale(W, K)
    r_value = float(r_value)
    r_peak = max(0.0, np.pi / (4.0 * theta_g) - 0.5)
    r_eval = min(r_value, r_peak)
    return float(np.sin((2.0 * r_eval + 1.0) * theta_g) ** 2)


def case_grover_scale(ctx):
    """Return r_Grover for a case context."""
    return grover_iteration_scale(ctx["W"], ctx["K"])


def case_time_limit(ctx):
    """Return the active CTQW scan time limit for one case."""
    # Si la cota esta activada, el paseo no puede usar mas tiempo que Grover.
    if V7_ENFORCE_GROVER_TIME_CAP:
        return case_grover_scale(ctx)
    return float(V7_DEFAULT_T_MAX)


def case_time_steps(ctx):
    """Return the number of sampled time points for one case."""
    # Conservamos resolucion alta incluso cuando t_max es pequeno.
    return int(V7_T_STEPS)

def valid_geometry_summary(ctx, A):
    """Summarise how the valid set sits inside the window-start graph."""
    # Contamos aristas internas validas y aristas de frontera valid-invalid.
    valid = set(ctx["valid_indices"])
    internal_valid_edges = 0
    boundary_edges = 0
    total_edges = 0
    degrees = A @ np.ones(A.shape[0])
    for i in range(ctx["W"]):
        for j in range(i + 1, ctx["W"]):
            if A[i, j] == 0:
                continue
            total_edges += 1
            if i in valid and j in valid:
                internal_valid_edges += 1
            elif (i in valid) != (j in valid):
                boundary_edges += 1
    valid_degrees = degrees[list(valid)] if valid else np.array([0.0])
    invalid = [i for i in range(ctx["W"]) if i not in valid]
    invalid_degrees = degrees[invalid] if invalid else np.array([0.0])
    return {
        "valid_fraction": ctx["K"] / ctx["W"],
        "internal_valid_edges": int(internal_valid_edges),
        "boundary_edges": int(boundary_edges),
        "total_edges": int(total_edges),
        "mean_valid_degree": float(np.mean(valid_degrees)),
        "mean_invalid_degree": float(np.mean(invalid_degrees)),
        "degree_bias_valid_minus_invalid": float(np.mean(valid_degrees) - np.mean(invalid_degrees)),
    }


## Lambda Scan for the Reference Case

The defect strength controls how strongly valid nodes are energetically separated from the rest of the graph. Here we scan $\lambda$ for case `02_1d_main_reference`, record the best success probability and best time for each value, and mark the optimal $\lambda^*$.


In [29]:
CASE02 = next(ctx for ctx in CASE_CONTEXTS if ctx["name"] == "02_1d_main_reference")
A_case02 = build_adjacency_matrix(CASE02["starts"], CASE02["N"])
lambda_values_case02 = v7_lambda_grid(CASE02["W"])
lambda_scan_case02 = scan_lambda(
    A_case02,
    CASE02["valid_indices"],
    CASE02["W"],
    lambda_values=lambda_values_case02,
    t_max=case_time_limit(CASE02),
    t_steps=case_time_steps(CASE02),
    hopping_sign=-1.0,
    peak_tolerance=V7_PEAK_TOLERANCE,
)
best_lam_case02, best_t_scan_case02, best_p_scan_case02 = best_lambda_from_scan(lambda_scan_case02)
r_grover_case02 = case_grover_scale(CASE02)

scan_lams = np.array([row[0] for row in lambda_scan_case02])
scan_tstars = np.array([row[1] for row in lambda_scan_case02])
scan_pstars = np.array([row[2] for row in lambda_scan_case02])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(scan_lams, scan_pstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[0].axhline(CASE02["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
axes[0].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[0].set_xlabel("lambda")
axes[0].set_ylabel("P_valid(t*)")
axes[0].set_title("Corrected CTQW sign: -A-lambdaP")
axes[0].legend(fontsize=10)

axes[1].plot(scan_lams, scan_tstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[1].axhline(r_grover_case02, color=INVALID_COLOR, linestyle="--", linewidth=1.5, label="r_Grover")
axes[1].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[1].set_xlabel("lambda")
axes[1].set_ylabel("earliest near-optimal t_star")
axes[1].set_title(f"Time selected within {V7_PEAK_TOLERANCE:.2f} of peak")
axes[1].legend(fontsize=10)

fig.suptitle("Lambda scan — 02_1d_main_reference", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_lambda_scan_case02")
print(f"Best lambda for case 02: lambda*={best_lam_case02:.6f}, t*={best_t_scan_case02:.6f}, P_valid={best_p_scan_case02:.6f}")


Best lambda for case 02: lambda*=1.288063, t*=1.568184, P_valid=0.788211


## Time Evolution Curve

For the optimal defect strength found above, we plot the full success-probability curve $P_{valid}(t)$. We also overlay the free walk with $\lambda=0$ to show why the defect is needed: without the local energy perturbation, the valid windows are not spectrally singled out.


In [30]:
rep_ctx = CASE02
rep_result = run_qw_case(rep_ctx, lam=best_lam_case02, gamma=1.0, hopping_sign=-1.0, t_max=case_time_limit(rep_ctx), t_steps=case_time_steps(rep_ctx), peak_tolerance=V7_PEAK_TOLERANCE)
free_case02 = free_walk_curve(rep_ctx, rep_result["A"], t_max=case_time_limit(rep_ctx), t_steps=case_time_steps(rep_ctx), hopping_sign=-1.0)
t_free, p_free = free_case02["t_values"], free_case02["p_curve"]

fig, ax = plt.subplots(figsize=(10, 5))
draw_time_evolution(ax, rep_result, free_curve=(t_free, p_free))
ax.set_title(f"Time evolution — {rep_ctx['name']}")
fig.tight_layout()
save_qw_figure(fig, f"v7_time_evolution_{qw_slug(rep_ctx['name'])}")


02_1d_main_reference | H=-A-lambdaP | W=7 | K=2 | lam*=1.2881 | t*=1.5682 | P_valid=0.7882 | vs Grover r=1.4693


## Eigenvalue Spectrum

The Hamiltonian is Hermitian, so its eigenvalues are real. A successful defect walk is associated with bound-state eigenvalues below the free-walk bulk band. These isolated eigenvalues explain why amplitude can localise around valid nodes.


In [31]:

fig, ax = plt.subplots(figsize=(10, 5))
draw_spectrum(ax, rep_result)
ax.set_title(f"Eigenvalue spectrum — {rep_ctx['name']}")
fig.tight_layout()
save_qw_figure(fig, f"v7_spectrum_{qw_slug(rep_ctx['name'])}")


## Optimal Probability Distribution

This bar chart shows the probability of measuring each candidate start index at the optimal time. Valid starts are green, invalid starts are red, and the dashed line is the uniform $1/W$ baseline.


In [32]:

fig, ax = plt.subplots(figsize=(10, 5))
draw_distribution(ax, rep_result)
ax.set_title(f"Optimal distribution — {rep_ctx['name']}, P_valid={rep_result['P_valid']:.4f}")
fig.tight_layout()
save_qw_figure(fig, f"v7_distribution_{qw_slug(rep_ctx['name'])}")


## 2x2 Summary Dashboard

The dashboard combines the key diagnostics for the representative case: time evolution, spectrum, optimal distribution, and lambda scan. This mirrors the compact comparison style used in V4.


In [33]:

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

draw_time_evolution(axes[0, 0], rep_result, free_curve=(t_free, p_free))
axes[0, 0].set_title("Time evolution")

draw_spectrum(axes[0, 1], rep_result)
axes[0, 1].set_title("Eigenvalue spectrum")

draw_distribution(axes[1, 0], rep_result)
axes[1, 0].set_title("Optimal distribution")

axes[1, 1].plot(scan_lams, scan_pstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[1, 1].axhline(rep_ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
axes[1, 1].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[1, 1].set_xlabel("lambda")
axes[1, 1].set_ylabel("P_valid(t*)")
axes[1, 1].set_title("Lambda scan")
axes[1, 1].legend(fontsize=9)

fig.suptitle(f"QW defect summary — {rep_ctx['name']}", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, f"v7_summary_{qw_slug(rep_ctx['name'])}")


## Cross-Case Comparison

We now apply the same lambda-scan procedure to all ten V4 cases. For each case, we choose the scanned $\lambda$ with the highest $P_{valid}(t^*)$, run the exact circuit, and compare the final success probability to the uniform baseline and Grover iteration scale.


In [34]:
ALL_QW_RESULTS = []
ALL_LAMBDA_SCANS = {}
ALL_OLD_SIGN_SCANS = {}

for ctx in CASE_CONTEXTS:
    A_ctx = build_adjacency_matrix(ctx["starts"], ctx["N"])
    lambda_values = v7_lambda_grid(ctx["W"])
    scan_rows = scan_lambda(
        A_ctx,
        ctx["valid_indices"],
        ctx["W"],
        lambda_values=lambda_values,
        t_max=case_time_limit(ctx),
        t_steps=case_time_steps(ctx),
        hopping_sign=-1.0,
        peak_tolerance=V7_PEAK_TOLERANCE,
    )
    ALL_LAMBDA_SCANS[ctx["name"]] = scan_rows
    best_lam, _best_t, _best_p = best_lambda_from_scan(scan_rows, p_tolerance=V7_PEAK_TOLERANCE)
    result = run_qw_case(
        ctx, lam=best_lam, gamma=1.0, hopping_sign=-1.0, t_max=case_time_limit(ctx), t_steps=case_time_steps(ctx),
        peak_tolerance=V7_PEAK_TOLERANCE,
    )

    free = free_walk_curve(ctx, A_ctx, t_max=case_time_limit(ctx), t_steps=case_time_steps(ctx), hopping_sign=-1.0)
    diagnostic_lambdas = v7_diagnostic_lambda_grid(ctx["W"])
    old_rows = scan_lambda_with_builder(
        A_ctx,
        ctx["valid_indices"],
        ctx["W"],
        diagnostic_lambdas,
        build_old_sign_hamiltonian,
        t_max=case_time_limit(ctx),
        t_steps=case_time_steps(ctx),
        peak_tolerance=V7_PEAK_TOLERANCE,
    )
    ALL_OLD_SIGN_SCANS[ctx["name"]] = old_rows
    old_lam, old_t, old_p = best_lambda_from_scan(old_rows, p_tolerance=V7_PEAK_TOLERANCE)
    geom = valid_geometry_summary(ctx, A_ctx)

    r_grover = case_grover_scale(ctx)
    p_grover = grover_success_probability(ctx["W"], ctx["K"], r_grover)

    result.update({
        "time_cap_enabled": bool(V7_ENFORCE_GROVER_TIME_CAP),
        "time_limit": case_time_limit(ctx),
        "P_Grover_theory": p_grover,
        "r_Grover": r_grover,
        "P_free": free["P_free"],
        "t_free": free["t_star"],
        "free_t_values": free["t_values"],
        "free_curve": free["p_curve"],
        "P_gain_over_uniform": result["P_valid"] - result["P_uniform"],
        "P_gain_over_free": result["P_valid"] - free["P_free"],
        "lambda_at_lower_bound": abs(result["lam"] - float(lambda_values[0])) < 1e-12,
        "old_sign_lambda": old_lam,
        "old_sign_t": old_t,
        "P_old_sign": old_p,
        "P_gain_over_old_sign": result["P_valid"] - old_p,
        **geom,
    })
    ALL_QW_RESULTS.append(result)

case_labels = [res["case"].split("_", 1)[0] for res in ALL_QW_RESULTS]
x = np.arange(len(ALL_QW_RESULTS))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

uniform_vals = np.array([res["P_uniform"] for res in ALL_QW_RESULTS])
qw_vals = np.array([res["P_valid"] for res in ALL_QW_RESULTS])
axes[0].bar(x - width/2, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="P_uniform")
axes[0].bar(x + width/2, qw_vals, width=width, color=VALID_COLOR, alpha=0.9, label="P_valid corrected QW")
for xi, value in zip(x, qw_vals):
    axes[0].text(xi + width/2, value + 0.015, f"{value:.2f}", ha="center", va="bottom", fontsize=9, rotation=90)
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels)
axes[0].set_ylim(0.0, min(1.15, max(1.0, qw_vals.max() + 0.12)))
axes[0].set_xlabel("case")
axes[0].set_ylabel("success probability")
axes[0].set_title("Uniform baseline vs corrected QW")
axes[0].legend(fontsize=10)

r_grover_vals = np.array([res["r_Grover"] for res in ALL_QW_RESULTS])
t_star_vals = np.array([res["t_star"] for res in ALL_QW_RESULTS])
axes[1].plot(x, r_grover_vals, marker="o", color=INVALID_COLOR, linewidth=2, label="r_Grover")
axes[1].plot(x, t_star_vals, marker="o", color=VALID_COLOR, linewidth=2, label="t_star")
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels)
axes[1].set_xlabel("case")
axes[1].set_ylabel("time / iteration scale")
axes[1].set_title("Grover scale vs corrected QW time")
axes[1].legend(fontsize=10)

fig.tight_layout()
save_qw_figure(fig, "v7_cross_case_comparison")


01_1d_tiny_single_gap | H=-A-lambdaP | W=5 | K=1 | lam*=1.9713 | t*=2.2353 | P_valid=0.6259 | vs Grover r=1.7562
02_1d_main_reference | H=-A-lambdaP | W=7 | K=2 | lam*=1.2881 | t*=1.5682 | P_valid=0.7882 | vs Grover r=1.4693
03_1d_two_free_regions | H=-A-lambdaP | W=8 | K=3 | lam*=0.9304 | t*=1.9019 | P_valid=0.8670 | vs Grover r=1.2825
04_1d_clustered_medium | H=-A-lambdaP | W=14 | K=4 | lam*=2.4124 | t*=0.9796 | P_valid=0.6121 | vs Grover r=1.4693
05_1d_long_clustered_blocks | H=-A-lambdaP | W=29 | K=11 | lam*=1.3463 | t*=1.9280 | P_valid=0.7145 | vs Grover r=1.2752
06_2d_tiny_corner_block | H=-A-lambdaP | W=4 | K=1 | lam*=3.2114 | t*=0.8539 | P_valid=0.7953 | vs Grover r=1.5708
07_2d_small_diagonal_block | H=-A-lambdaP | W=9 | K=1 | lam*=3.1184 | t*=1.7782 | P_valid=0.7033 | vs Grover r=2.3562
08_2d_medium_clustered_obstacles | H=-A-lambdaP | W=16 | K=9 | lam*=3.8421 | t*=0.4808 | P_valid=0.8380 | vs Grover r=1.0472
09_2d_rectangular_window | H=-A-lambdaP | W=20 | K=8 | lam*=3.8017 

## Per-Case Dashboards

The previous cells focused on the representative 1D case. This cell creates the same summary diagnostics for every case: lambda scan, time evolution with free-walk overlay, spectrum, and final probability distribution. These figures make it easier to see whether each instance is improved by the defect or mostly by free graph evolution.


In [35]:
for result in ALL_QW_RESULTS:
    ctx = result["ctx"]
    slug = qw_slug(ctx["name"])
    scan_rows = ALL_LAMBDA_SCANS[ctx["name"]]
    lams = np.array([row[0] for row in scan_rows])
    pstars = np.array([row[2] for row in scan_rows])
    tstars = np.array([row[1] for row in scan_rows])

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    axes[0, 0].plot(lams, pstars, marker="o", color=VALID_COLOR, linewidth=2)
    axes[0, 0].axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    axes[0, 0].axhline(result["P_free"], color="black", linestyle="-.", linewidth=1.3, label="free max")
    axes[0, 0].axvline(result["lam"], color="black", linestyle=":", linewidth=1.5, label=f"lambda*={result['lam']:.3f}")
    axes[0, 0].set_xlabel("lambda")
    axes[0, 0].set_ylabel("P_valid(t*)")
    axes[0, 0].set_title("Corrected -A-lambdaP scan")
    axes[0, 0].legend(fontsize=8)

    draw_time_evolution(axes[0, 1], result, free_curve=(result["free_t_values"], result["free_curve"]))
    axes[0, 1].set_title("Time evolution")

    draw_distribution(axes[1, 0], result)
    axes[1, 0].set_title(f"Distribution, P={result['P_valid']:.4f}")

    draw_spectrum(axes[1, 1], result)
    axes[1, 1].set_title("Spectrum")

    fig.suptitle(f"Corrected QW defect per-case summary — {ctx['name']}", fontsize=14)
    fig.tight_layout()
    save_qw_figure(fig, f"v7_case_summary_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(lams, pstars, marker="o", color=VALID_COLOR, linewidth=2, label="corrected -A-lambdaP")
    ax.axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.axhline(result["P_free"], color="black", linestyle="-.", linewidth=1.4, label="free walk max")
    ax.axvline(result["lam"], color="black", linestyle=":", linewidth=1.5, label=f"lambda*={result['lam']:.3f}")
    ax.set_xlabel("lambda")
    ax.set_ylabel("P_valid(t*)")
    ax.set_title(f"Lambda scan — {ctx['name']}")
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_qw_figure(fig, f"v7_lambda_scan_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_time_evolution(ax, result, free_curve=(result["free_t_values"], result["free_curve"]))
    ax.set_title(f"Time evolution — {ctx['name']}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_time_evolution_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_distribution(ax, result)
    ax.set_title(f"Optimal distribution — {ctx['name']}, P_valid={result['P_valid']:.4f}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_distribution_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_spectrum(ax, result)
    ax.set_title(f"Eigenvalue spectrum — {ctx['name']}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_spectrum_{slug}")

print(f"Saved corrected per-case V7 analysis figures for {len(ALL_QW_RESULTS)} cases.")


Saved corrected per-case V7 analysis figures for 10 cases.


## All-Case Diagnostic Figures

These all-case plots compare the uniform baseline, the free walk, the corrected `-A - lambda P` defect walk, and the previous `+A - lambda P` convention. The goal is to verify that the fix improves the probabilities and reduces the chosen times by avoiding late recurrences.


In [36]:

case_labels = [res["case"].split("_", 1)[0] for res in ALL_QW_RESULTS]
x = np.arange(len(ALL_QW_RESULTS))

uniform_vals = np.array([res["P_uniform"] for res in ALL_QW_RESULTS])
free_vals = np.array([res["P_free"] for res in ALL_QW_RESULTS])
corrected_vals = np.array([res["P_valid"] for res in ALL_QW_RESULTS])
old_vals = np.array([res["P_old_sign"] for res in ALL_QW_RESULTS])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
width = 0.2
axes[0].bar(x - 1.5*width, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="uniform")
axes[0].bar(x - 0.5*width, free_vals, width=width, color="black", alpha=0.65, label="free -A")
axes[0].bar(x + 0.5*width, old_vals, width=width, color=INVALID_COLOR, alpha=0.75, label="old +A-lambdaP")
axes[0].bar(x + 1.5*width, corrected_vals, width=width, color=VALID_COLOR, alpha=0.9, label="corrected -A-lambdaP")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels)
axes[0].set_ylabel("best P_valid")
axes[0].set_xlabel("case")
axes[0].set_title("Hamiltonian sign comparison")
axes[0].legend(fontsize=9)

axes[1].bar(x - width/2, corrected_vals - old_vals, width=width, color=VALID_COLOR, alpha=0.9, label="corrected - old")
axes[1].bar(x + width/2, corrected_vals - free_vals, width=width, color="#3498db", alpha=0.85, label="corrected - free")
axes[1].axhline(0.0, color="black", linewidth=1.0)
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels)
axes[1].set_ylabel("probability difference")
axes[1].set_xlabel("case")
axes[1].set_title("Improvement after the fix")
axes[1].legend(fontsize=9)
fig.tight_layout()
save_qw_figure(fig, "v7_hamiltonian_sign_comparison")

fig, axes = plt.subplots(2, 5, figsize=(14, 7), sharey=True)
for ax, result in zip(axes.ravel(), ALL_QW_RESULTS):
    scan_rows = ALL_LAMBDA_SCANS[result["case"]]
    lams = np.array([row[0] for row in scan_rows])
    pstars = np.array([row[2] for row in scan_rows])
    ax.plot(lams, pstars, color=VALID_COLOR, linewidth=1.8)
    ax.axhline(result["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.0)
    ax.axhline(result["P_free"], color="black", linestyle="-.", linewidth=1.0)
    ax.axvline(result["lam"], color="black", linestyle=":", linewidth=1.0)
    ax.set_title(result["case"].split("_", 1)[0], fontsize=11)
    ax.set_xlabel("lambda")
axes[0, 0].set_ylabel("P_valid(t*)")
axes[1, 0].set_ylabel("P_valid(t*)")
fig.suptitle("All lambda scans — corrected -A-lambdaP Hamiltonian", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_all_lambda_scans")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid_fraction = np.array([res["valid_fraction"] for res in ALL_QW_RESULTS])
degree_bias = np.array([res["degree_bias_valid_minus_invalid"] for res in ALL_QW_RESULTS])
gain_vs_uniform = corrected_vals - uniform_vals
gain_vs_free = corrected_vals - free_vals

axes[0].scatter(valid_fraction, gain_vs_uniform, s=80, color=VALID_COLOR)
for label, xf, yf in zip(case_labels, valid_fraction, gain_vs_uniform):
    axes[0].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[0].axhline(0.0, color="black", linewidth=1.0)
axes[0].set_xlabel("valid fraction K/W")
axes[0].set_ylabel("P_corrected - P_uniform")
axes[0].set_title("Corrected gain vs valid-set size")

axes[1].scatter(degree_bias, gain_vs_free, s=80, color="#3498db")
for label, xf, yf in zip(case_labels, degree_bias, gain_vs_free):
    axes[1].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[1].axhline(0.0, color="black", linewidth=1.0)
axes[1].set_xlabel("mean degree(valid) - mean degree(invalid)")
axes[1].set_ylabel("P_corrected - P_free")
axes[1].set_title("Corrected gain relative to free walk")
fig.tight_layout()
save_qw_figure(fig, "v7_gain_diagnostics")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
old_t = np.array([res["old_sign_t"] for res in ALL_QW_RESULTS])
corrected_t = np.array([res["t_star"] for res in ALL_QW_RESULTS])
axes[0].bar(x - width/2, old_t, width=width, color=INVALID_COLOR, alpha=0.75, label="old +A-lambdaP")
axes[0].bar(x + width/2, corrected_t, width=width, color=VALID_COLOR, alpha=0.9, label="corrected -A-lambdaP")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels)
axes[0].set_ylabel("selected t_star")
axes[0].set_xlabel("case")
axes[0].set_title("Selected walk time after earliest-peak rule")
axes[0].legend(fontsize=9)

axes[1].scatter(corrected_t, corrected_vals, s=80, color=VALID_COLOR)
for label, xf, yf in zip(case_labels, corrected_t, corrected_vals):
    axes[1].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[1].set_xlabel("corrected t_star")
axes[1].set_ylabel("P_valid")
axes[1].set_title("Probability-time tradeoff")
fig.tight_layout()
save_qw_figure(fig, "v7_time_reduction_diagnostics")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
grover_p = np.array([res["P_Grover_theory"] for res in ALL_QW_RESULTS])
grover_t = np.array([res["r_Grover"] for res in ALL_QW_RESULTS])
qw_p = np.array([res["P_valid"] for res in ALL_QW_RESULTS])
qw_t = np.array([res["t_star"] for res in ALL_QW_RESULTS])

axes[0].bar(x - width/2, grover_p, width=width, color=INVALID_COLOR, alpha=0.75, label="Grover theoretical")
axes[0].bar(x + width/2, qw_p, width=width, color=VALID_COLOR, alpha=0.9, label="V7 quantum walk")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels)
axes[0].set_ylim(0.0, 1.08)
axes[0].set_xlabel("case")
axes[0].set_ylabel("success probability")
axes[0].set_title("Probability face-off")
axes[0].legend(fontsize=9)

axes[1].bar(x - width/2, grover_t, width=width, color=INVALID_COLOR, alpha=0.75, label="Grover r_Grover")
axes[1].bar(x + width/2, qw_t, width=width, color=VALID_COLOR, alpha=0.9, label="V7 QW t*")
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels)
axes[1].set_xlabel("case")
axes[1].set_ylabel("time / iteration scale")
axes[1].set_title(
    "Time face-off" + (" (Grover cap ON)" if V7_ENFORCE_GROVER_TIME_CAP else " (Grover cap OFF)")
)
axes[1].legend(fontsize=9)
fig.suptitle("V7 quantum walk vs Grover theoretical", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_qw_vs_grover_faceoff")

print("Main diagnosis after fix:")
print("  1. The main Hamiltonian is now the CTQW search convention H=-A-lambdaP.")
print("  2. Lambda selection keeps probabilities within V7_PEAK_TOLERANCE of the best scan value and chooses the earliest peak.")
print("  3. The old +A-lambdaP convention remains only as a diagnostic baseline.")


Main diagnosis after fix:
  1. The main Hamiltonian is now the CTQW search convention H=-A-lambdaP.
  2. Lambda selection keeps probabilities within V7_PEAK_TOLERANCE of the best scan value and chooses the earliest peak.
  3. The old +A-lambdaP convention remains only as a diagnostic baseline.


## Comparison Table

The final table reports the main numerical quantities for every case: search-space size, number of valid windows, uniform baseline, optimal defect strength, optimal time, quantum-walk success probability, and Grover iteration scale. The same data are saved as CSV.


In [37]:

headers = [
    "case", "W", "K", "P_uniform", "lambda*", "t*", "P_valid(QW)",
    "time_cap_enabled", "time_limit", "P_Grover_theory", "P_free", "gain_vs_uniform", "gain_vs_free", "lambda_at_lower_bound",
    "P_old_sign", "lambda_old_sign", "t_old_sign", "gain_vs_old_sign", "r_Grover",
    "valid_fraction", "degree_bias_valid_minus_invalid",
]
rows = []
for res in ALL_QW_RESULTS:
    ctx = res["ctx"]
    rows.append({
        "case": res["case"],
        "W": ctx["W"],
        "K": ctx["K"],
        "P_uniform": res["P_uniform"],
        "lambda*": res["lam"],
        "t*": res["t_star"],
        "P_valid(QW)": res["P_valid"],
        "time_cap_enabled": res["time_cap_enabled"],
        "time_limit": res["time_limit"],
        "P_Grover_theory": res["P_Grover_theory"],
        "P_free": res["P_free"],
        "gain_vs_uniform": res["P_gain_over_uniform"],
        "gain_vs_free": res["P_gain_over_free"],
        "lambda_at_lower_bound": res["lambda_at_lower_bound"],
        "P_old_sign": res["P_old_sign"],
        "lambda_old_sign": res["old_sign_lambda"],
        "t_old_sign": res["old_sign_t"],
        "gain_vs_old_sign": res["P_gain_over_old_sign"],
        "r_Grover": res["r_Grover"],
        "valid_fraction": res["valid_fraction"],
        "degree_bias_valid_minus_invalid": res["degree_bias_valid_minus_invalid"],
    })

print(" | ".join(f"{h:>18s}" for h in headers))
print("-" * (21 * len(headers)))
for row in rows:
    print(
        f"{row['case']:>18s} | {row['W']:18d} | {row['K']:18d} | "
        f"{row['P_uniform']:18.6f} | {row['lambda*']:18.6f} | "
        f"{row['t*']:18.6f} | {row['P_valid(QW)']:18.6f} | "
        f"{str(row['time_cap_enabled']):>18s} | {row['time_limit']:18.6f} | "
        f"{row['P_Grover_theory']:18.6f} | {row['P_free']:18.6f} | {row['gain_vs_uniform']:18.6f} | "
        f"{row['gain_vs_free']:18.6f} | {str(row['lambda_at_lower_bound']):>18s} | "
        f"{row['P_old_sign']:18.6f} | {row['lambda_old_sign']:18.6f} | "
        f"{row['t_old_sign']:18.6f} | {row['gain_vs_old_sign']:18.6f} | "
        f"{row['r_Grover']:18.6f} | {row['valid_fraction']:18.6f} | "
        f"{row['degree_bias_valid_minus_invalid']:18.6f}"
    )

csv_path = V7_OUTPUT_DIR / "v7_results.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved {csv_path}")


              case |                  W |                  K |          P_uniform |            lambda* |                 t* |        P_valid(QW) |   time_cap_enabled |         time_limit |    P_Grover_theory |             P_free |    gain_vs_uniform |       gain_vs_free | lambda_at_lower_bound |         P_old_sign |    lambda_old_sign |         t_old_sign |   gain_vs_old_sign |           r_Grover |     valid_fraction | degree_bias_valid_minus_invalid
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
01_1d_tiny_single_gap |                  5 |                  1 |           0.200000 |           1.9712

## Bonus — Optional First-Order Trotter Approximation

This optional cell is disabled by default. It makes the V4 connection explicit by approximating

$$e^{-i(\gamma A-\lambda P_V)t}\approx\left(e^{-i\gamma A\Delta t}e^{i\lambda P_V\Delta t}\right)^r.$$

The diagonal phase layer applies a controlled phase to each classically known valid index. The mixer layer applies exact two-level rotations for every graph edge. Increasing the Trotter count should converge toward the exact dense `UnitaryGate` circuit.


In [38]:

TROTTER_ENABLED = False


def apply_phase_to_basis_state(qc, idx, basis_state, angle, label=None):
    """Apply exp(i*angle) to one computational basis state of idx."""
    # Convertimos el estado objetivo en |11...1>, aplicamos fase controlada y deshacemos.
    if abs(float(angle)) < 1e-15:
        return
    IDX = len(idx)
    zero_qubits = [q for bit, q in enumerate(idx) if ((int(basis_state) >> bit) & 1) == 0]
    for q in zero_qubits:
        qc.x(q)
    if IDX == 1:
        qc.p(float(angle), idx[0])
    else:
        gate = PhaseGate(float(angle), label=label).control(IDX - 1)
        qc.append(gate, list(idx[:-1]) + [idx[-1]])
    for q in reversed(zero_qubits):
        qc.x(q)


def two_level_rotation_full(W, IDX, a, b, angle):
    """Return embedded exp(-i angle (|a><b| + |b><a|)) on the idx register."""
    # La rotacion exacta actua solo en el subespacio generado por |a> y |b>.
    dim = 2**IDX
    U = np.eye(dim, dtype=complex)
    c = np.cos(float(angle))
    s = -1j * np.sin(float(angle))
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return U


def adjacency_edges(A):
    """Return undirected edges (a,b) from the upper triangle of A."""
    # Extraemos cada arista una sola vez.
    edges = []
    W = A.shape[0]
    for a in range(W):
        for b in range(a + 1, W):
            if abs(A[a, b]) > 1e-12:
                edges.append((a, b))
    return edges


def build_trotter_qw_circuit(ctx, lam, t_star, gamma=1.0, r_trotter=4):
    """Build the first-order Trotterised approximation to the defect walk."""
    # Preparamos el mismo registro idx y alternamos mixer local + fase de defectos.
    W = ctx["W"]
    IDX = ctx["IDX"]
    dt = float(t_star) / int(r_trotter)
    A = build_adjacency_matrix(ctx["starts"], ctx["N"])
    edges = adjacency_edges(A)

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)

    for layer in range(int(r_trotter)):
        for edge_id, (a, b) in enumerate(edges):
            U_edge = two_level_rotation_full(W, IDX, a, b, -gamma * dt)
            qc.append(UnitaryGate(U_edge, label=f"mix[{layer},{edge_id}]") , list(idx))
        for valid_id in ctx["valid_indices"]:
            apply_phase_to_basis_state(qc, idx, valid_id, lam * dt, label=f"def[{layer},{valid_id}]")
        qc.barrier(label=f"Trotter r={layer + 1}")
    return qc


if TROTTER_ENABLED:
    trotter_counts = [1, 2, 4, 8, 16]
    exact_p = rep_result["P_valid"]
    trotter_probs = []
    for r_trotter in trotter_counts:
        qc_trot = build_trotter_qw_circuit(
            rep_ctx,
            lam=rep_result["lam"],
            t_star=rep_result["t_star"],
            gamma=rep_result["gamma"],
            r_trotter=r_trotter,
        )
        sv_trot = Statevector.from_instruction(qc_trot)
        probs_trot = index_probs_from_statevector(sv_trot, rep_ctx["W"], rep_ctx["IDX"])
        p_trot = float(np.sum(probs_trot[rep_ctx["valid_indices"]]))
        trotter_probs.append(p_trot)
        print(f"r_trotter={r_trotter:2d} | P_valid={p_trot:.6f} | exact={exact_p:.6f}")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(trotter_counts, trotter_probs, marker="o", color=VALID_COLOR, linewidth=2, label="Trotter")
    ax.axhline(exact_p, color="black", linestyle="--", linewidth=1.5, label="exact corrected UnitaryGate")
    ax.set_xscale("log", base=2)
    ax.set_xticks(trotter_counts)
    ax.set_xticklabels([str(r) for r in trotter_counts])
    ax.set_xlabel("Trotter steps r")
    ax.set_ylabel("P_valid")
    ax.set_title(f"Trotter convergence — {rep_ctx['name']}")
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_qw_figure(fig, f"v7_trotter_convergence_{qw_slug(rep_ctx['name'])}")
else:
    print("TROTTER_ENABLED = False. Set it to True to run the optional V4-to-QW Trotter comparison.")


TROTTER_ENABLED = False. Set it to True to run the optional V4-to-QW Trotter comparison.
